# Engine & Worker Speed Comparison

**Question:** how much faster did the simulation pipeline get, and *where* did the
speedup actually come from? Two things changed between June 8 and June 9:

1. **The engine** can be the pure-Python reference (`MoranProcess`) or the C++ core
   (`CppMoranProcess`, ~1800x faster in the inner loop).
2. **The worker** was rewritten: the per-repeat loop moved *inside* the engine
   (`run_repeats`, one language-boundary crossing per task instead of `2*n_repeats`),
   and the worker's import footprint was cut (lazy matplotlib/networkx, a lean
   `GraphCore`), dropping cold-start module loads substantially.

We ran the **same 19-graph zoo** (same `batch_seed=12345`, so bit-identical graphs)
at the same `r` values with `n_repeats=1000`, across a clean **2x2** design:

| | **baseline** (Jun 8) | **repeats + lazy imports** (Jun 9) |
|---|---|---|
| **python** | `2026-06-08_test-engines-python` | `2026-06-09_test-repeats-engines-python` |
| **cpp**    | `2026-06-08_test-engines-cpp`    | `2026-06-09_test-repeats-engines-cpp`    |

Every batch is 19 graphs x 4 r-values x 1000 repeats = 76,000 simulations, spread
over 800 LSF array jobs. Because the design is identical, any wall-clock difference
is attributable to the engine or the worker rewrite, not to the workload.

### What we measure, and why two clocks matter

- **Wall-clock per job** (`run_sec`, from the LSF `.out` logs): the real elapsed
  time a job occupied a core. This *includes* Python startup, NFS imports, shard
  load and Parquet IO. It is what determines time-to-result and CPU-hour cost.
- **Pure simulation time** (`duration`, measured by the worker around the engine
  call only): isolates true engine throughput from the fixed per-job overhead.

The gap between these two clocks is the whole story below.

In [ ]:
%load_ext autoreload
%autoreload 2
%cd /home/labs/pilpel/matanyaw/moran-process

import sys
sys.path.insert(0, 'src')

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
from pathlib import Path

from moran_process.analysis.analysis_utils import load_batch_info
from moran_process.analysis.batch_speed_report import batch_speed_report, _parse_all_log_times

ROOT = Path.cwd()
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 30)

## The four batches

We tag each batch with its `engine` and `variant` so the comparison table and plots
can group by them. `variant='baseline'` is the Jun-8 worker; `variant='repeats+lazy'`
is the Jun-9 rewrite.

In [ ]:
BATCHES = [
    {'name': '2026-06-08_test-engines-python',          'engine': 'python', 'variant': 'baseline'},
    {'name': '2026-06-08_test-engines-cpp',             'engine': 'cpp',    'variant': 'baseline'},
    {'name': '2026-06-09_test-repeats-engines-python',  'engine': 'python', 'variant': 'repeats+lazy'},
    {'name': '2026-06-09_test-repeats-engines-cpp',     'engine': 'cpp',    'variant': 'repeats+lazy'},
]
for b in BATCHES:
    b['label'] = f"{b['engine']}/{b['variant']}"

# Sanity-check the design is really held fixed across batches.
for b in BATCHES:
    info = load_batch_info(ROOT / 'simulation_data' / b['name'])
    print(f"{b['label']:24} seed={info['batch_seed']}  n_graphs={info['n_graphs']}  "
          f"r={info['r_values']}  reps={info['n_repeats']}  engine={info['engine']}")

## Collecting per-batch speed stats

For each batch we join two sources on `job_id`:

- the **result Parquet** (`full_results.parquet`) -> per-job `steps` and `duration`
  (the worker-timed pure-simulation seconds), and
- the **LSF logs** (`logs/*.out`, parsed by `_parse_all_log_times`) -> per-job
  `run_sec` (wall-clock) and peak/avg RAM.

From the per-job join we derive the headline numbers. Two throughput definitions are
computed deliberately:

- `sim_thru_M` = total steps / total **sim** seconds -> what the engine can do.
- `wall_thru_k` = total steps / total **wall** seconds -> what you actually get.

Their ratio is how badly fixed per-job overhead masks the engine.

In [ ]:
def collect_batch_stats(batch):
    """Return one row of headline speed metrics for a tagged batch dict."""
    bd = ROOT / 'simulation_data' / batch['name']

    # Only the three speed-relevant columns; stays light even for big batches.
    df = (pl.scan_parquet(bd / 'full_results.parquet')
            .select(['job_id', 'steps', 'duration'])
            .collect().to_pandas())
    job_steps = df.groupby('job_id')['steps'].sum().rename('total_steps')
    job_sim   = df.groupby('job_id')['duration'].sum().rename('sim_sec')

    logs = _parse_all_log_times(bd / 'logs').set_index('job_id')
    j = logs.join(job_steps).join(job_sim).dropna(subset=['run_sec', 'total_steps'])

    total_steps = j['total_steps'].sum()
    run_s       = j['run_sec'].sum()
    sim_s       = j['sim_sec'].sum()
    return {
        'label':         batch['label'],
        'engine':        batch['engine'],
        'variant':       batch['variant'],
        'n_jobs':        len(j),
        'steps_M':       total_steps / 1e6,
        'avg_wall_s':    j['run_sec'].mean(),
        'max_wall_s':    j['run_sec'].max(),       # ~ time-to-result (slowest job)
        'cpu_h':         run_s / 3600,             # summed wall = CPU-hour cost
        'sim_s_total':   sim_s,
        'sim_pct':       100 * sim_s / run_s,      # fraction of wall actually simulating
        'sim_thru_M':    total_steps / sim_s  / 1e6,
        'wall_thru_k':   total_steps / run_s  / 1e3,
        'peak_ram_avg':  j['max_mem_mb'].mean(),
        'peak_ram_max':  j['max_mem_mb'].max(),
    }

summary = pd.DataFrame([collect_batch_stats(b) for b in BATCHES]).set_index('label')
summary.round(2)

## Reading the headline table

Three comparisons matter. The cells below compute each as an explicit ratio so the
numbers, not the prose, make the point.

1. **Engine swap (cpp vs python), inner-loop speed.** `sim_thru_M` should jump by
   ~1000x+ for cpp: that is the raw C++ win in the hot loop.
2. **Engine swap, wall-clock.** `avg_wall_s` and `wall_thru_k` should be *almost
   unchanged* by the engine. If true, the engine speed is being masked by overhead.
3. **Worker rewrite (baseline vs repeats+lazy).** `avg_wall_s`, `cpu_h` and
   `peak_ram_*` should drop together: this is the lighter import footprint paying off,
   and it helps *both* engines equally.

In [ ]:
def ratio(metric, num_label, den_label, fmt='{:.1f}x'):
    num, den = summary.loc[num_label, metric], summary.loc[den_label, metric]
    return fmt.format(num / den)

print('1) ENGINE inner-loop speed (sim-only throughput), cpp vs python')
print(f"   baseline      : cpp is {ratio('sim_thru_M','cpp/baseline','python/baseline','{:.0f}x')} faster in-engine")
print(f"   repeats+lazy  : cpp is {ratio('sim_thru_M','cpp/repeats+lazy','python/repeats+lazy','{:.0f}x')} faster in-engine")
print()
print('2) Same engine swap, but at the WALL CLOCK (avg seconds per job)')
print(f"   baseline      : cpp/python wall ratio = {ratio('avg_wall_s','cpp/baseline','python/baseline','{:.2f}x')}  (~1.0 => engine hidden)")
print(f"   repeats+lazy  : cpp/python wall ratio = {ratio('avg_wall_s','cpp/repeats+lazy','python/repeats+lazy','{:.2f}x')}")
print()
print('3) WORKER rewrite (baseline -> repeats+lazy), same engine')
print(f"   python wall   : {ratio('avg_wall_s','python/baseline','python/repeats+lazy')} faster   |  RAM {ratio('peak_ram_avg','python/baseline','python/repeats+lazy')} smaller")
print(f"   cpp    wall   : {ratio('avg_wall_s','cpp/baseline','cpp/repeats+lazy')} faster   |  RAM {ratio('peak_ram_avg','cpp/baseline','cpp/repeats+lazy')} smaller")

## Per-batch detailed report

`batch_speed_report` prints the full per-batch breakdown (wall throughput, the
sim-only engine throughput, the overhead-masking factor, RAM) and draws the run-time
/ speed / RAM histograms. Running it for all four makes the distribution shape
visible, not just the means.

In [ ]:
for b in BATCHES:
    bd = ROOT / 'simulation_data' / b['name']
    df_speed = (pl.scan_parquet(bd / 'full_results.parquet')
                  .select(['job_id', 'steps', 'duration'])
                  .collect().to_pandas())
    batch_speed_report(b['label'], df_speed, bd)
    print()

## Cross-batch visual comparison

Four grouped-bar panels, one per metric, with python vs cpp side by side for each
worker variant. The log scale on sim-time/throughput is essential: linear axes would
flatten the ~1800x engine gap into an invisible sliver.

In [ ]:
variants = ['baseline', 'repeats+lazy']
engines  = ['python', 'cpp']
colors   = {'python': 'tab:blue', 'cpp': 'tab:orange'}
x = np.arange(len(variants))
w = 0.38

def get(engine, variant, metric):
    return summary.loc[f'{engine}/{variant}', metric]

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle('Engine x Worker-variant speed comparison (same 76k-sim workload)', fontsize=13)

panels = [
    ('avg_wall_s',  'Avg wall-clock per job (s)', False, 'lower = faster time-to-result'),
    ('peak_ram_avg','Avg peak RAM per job (MB)',  False, 'lower = lighter import footprint'),
    ('sim_s_total', 'Total pure-sim time (s, all jobs)', True, 'engine-only; log scale'),
    ('sim_thru_M',  'Sim throughput (M steps/s)', True, 'raw engine speed; log scale'),
]

for ax, (metric, title, logy, sub) in zip(axes.flat, panels):
    for k, engine in enumerate(engines):
        vals = [get(engine, v, metric) for v in variants]
        bars = ax.bar(x + (k - 0.5) * w, vals, w, label=engine, color=colors[engine])
        ax.bar_label(bars, fmt='%.2g', padding=2, fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels(variants)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel(sub, fontsize=9, color='gray')
    if logy:
        ax.set_yscale('log')
    ax.legend(title='engine', fontsize=9)

plt.tight_layout()
plt.show()

### Where each job's wall-clock actually goes

The bar below splits average wall-clock per job into **pure simulation** vs
**everything else** (startup + imports + IO). For the cpp engine the simulation slice
is invisible: all 800 cpp jobs together simulate for under a second, while each job
still pays the full startup tax. This is the picture of a *startup-bound* workload.

In [ ]:
labels = [b['label'] for b in BATCHES]
avg_wall = summary['avg_wall_s'].reindex(labels).values
avg_sim  = (summary['sim_s_total'] / summary['n_jobs']).reindex(labels).values
avg_over = avg_wall - avg_sim

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(labels, avg_sim, label='pure simulation (engine)', color='seagreen')
ax.bar(labels, avg_over, bottom=avg_sim, label='overhead (startup + imports + IO)', color='lightgray')
for i, (s, o) in enumerate(zip(avg_sim, avg_over)):
    ax.text(i, s + o + 0.5, f'sim {s:.2f}s\n{100*s/(s+o):.2g}% of wall', ha='center', va='bottom', fontsize=8)
ax.set_ylabel('Avg seconds per job')
ax.set_title('Per-job wall-clock: simulation vs fixed overhead')
ax.legend()
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## Conclusion

Reading the four batches together:

- **The C++ engine is ~1800x faster in the inner loop** (`sim_thru_M`: ~80-89 M
  steps/s vs ~0.04-0.05 M for Python). That win is real and it is what the core was
  built for.
- **At this workload it is almost entirely masked.** Each job simulates only ~86k
  steps, so cpp jobs spend ~0.00% of their wall-clock simulating (all 800 finish in
  ~0.8 s combined). The wall throughput of cpp and python is therefore the *same*:
  both jobs are dominated by the ~18-50 s fixed startup floor (Python + NFS imports +
  IO), not by the math.
- **The lever that actually moved wall-clock was the worker rewrite, not the engine.**
  Lighter imports cut avg wall-clock from ~50 s to ~18 s (~2.7x), halved CPU-hours,
  and roughly halved peak RAM (~130 -> ~60 MB) -- and it helped *both* engines
  identically, which is the fingerprint of an overhead fix rather than a compute fix.

**Implication for real runs.** To convert the C++ engine's 1800x into wall-clock,
each job must do enough simulation that `sim_pct` climbs well above the startup floor:
bigger graphs, more repeats per job, or fewer/fatter jobs. Until per-job sim time
exceeds the ~18 s import floor, the pipeline is startup-bound and the engine choice
barely shows up at the wall clock -- so the next optimization target is amortizing
startup (more work per job), not the inner loop.